베터리에 큰 영향을 주는 요인을 주는 요인 찾기


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import matplotlib.pyplot as plt
import matplotlib as mpl
import platform
import ast

df1= pd.read_csv("metadata.csv")

In [2]:
df = df1.copy()

In [3]:
plt.rcParams['font.family'] = 'Malgun Gothic' # For Windows 
%matplotlib inline

df = pd.read_csv("metadata.csv", encoding='utf-8')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   type                 7565 non-null   str  
 1   start_time           7565 non-null   str  
 2   ambient_temperature  7565 non-null   int64
 3   battery_id           7565 non-null   str  
 4   test_id              7565 non-null   int64
 5   uid                  7565 non-null   int64
 6   filename             7565 non-null   str  
 7   Capacity             2794 non-null   str  
 8   Re                   1956 non-null   str  
 9   Rct                  1956 non-null   str  
dtypes: int64(3), str(7)
memory usage: 591.1 KB


In [5]:
import pandas as pd

def check_missing(df, top_n=None):
    missing_df = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': df.isna().mean()*100
    }).sort_values('missing_count', ascending=False)
    
    if top_n:
        return missing_df.head(top_n)
    
    return missing_df


In [6]:
check_missing(df)

,missing_count,missing_ratio
Rct,5609,74.144085
Re,5609,74.144085
Capacity,4771,63.066755
type,0,0.000000
start_time,0,0.000000
ambient_temperature,0,0.000000
uid,0,0.000000
test_id,0,0.000000
battery_id,0,0.000000
filename,0,0.000000


Re(전해질 저항(Ω)) 결축 5609개 ,
Rct(충전 전달 저항(Ω)) 결축 5609개,
Capacity(배터리 현재 용량(Ah)) 결측 4771개

Re , Rct, Capacity = 내용은 숫자인데 타입은 문자형으로 찍힘

start_time(테스트 시작 시간) datetime으로 변환 필요

test_id(실험 ID) 는 0 ~615 

In [7]:
df['battery_id'].value_counts().sum()

np.int64(7565)

In [8]:
df['Capacity'] = pd.to_numeric(df['Capacity'], errors='coerce')
df['Re'] = pd.to_numeric(df['Re'], errors='coerce')
df['Rct'] = pd.to_numeric(df['Rct'], errors='coerce')

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   type                 7565 non-null   str    
 1   start_time           7565 non-null   str    
 2   ambient_temperature  7565 non-null   int64  
 3   battery_id           7565 non-null   str    
 4   test_id              7565 non-null   int64  
 5   uid                  7565 non-null   int64  
 6   filename             7565 non-null   str    
 7   Capacity             2769 non-null   float64
 8   Re                   1947 non-null   float64
 9   Rct                  1947 non-null   float64
dtypes: float64(3), int64(3), str(4)
memory usage: 591.1 KB


형테 이상해서 pd.to_datetime 안됨
리스트 분해해서 형태 재정비부터


In [10]:
df['start_time']

0       [2010.       7.      21.      15.       0.    ...
1       [2010.       7.      21.      16.      53.    ...
2       [2010.       7.      21.      17.      25.    ...
3                         [2010    7   21   20   31    5]
4       [2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...
                              ...                        
7560    [2010.       9.      30.       7.      36.    ...
7561    [2010.       9.      30.       8.       8.    ...
7562    [2010.      9.     30.      8.     48.     54.25]
7563    [2010.       9.      30.      11.      50.    ...
7564    [2010.       9.      30.      12.      31.    ...
Name: start_time, Length: 7565, dtype: str

In [11]:
def clear_time(x):
    #사용할 리스트
    nums = []
    #임시 저장용
    current = ''
    
# 반복문으로 숫자만 뽑아내기
    for ch in x:
        #숫자형이거나 형태 다른 숫자 뽑아내기
        if ch.isdigit() or ch in ['.','_','e','E','+']:
            current += ch
        else:
            if current != '' :
                nums.append(current)
                current = ''
            
    #숫자 남아있을 수도 있으니 한번더 이스트에 append
    if current != '':
        nums.append(current)
    
    #연월일시분초 하면 최소 6자리, 그 이하는 버림
    if len(nums) <6 :
        return None
    
    #num리스트 숫자형으로 변환, 연월일시분초 하면 6개임으로 앞에 6개만 숫자변환
    try :
        nums = list(map(float, nums[:6]))
    except:
        return None
    
    #각 숫자에 의미부여
    year, month, day, hour, minute, second = nums
    
    # 이상치 제거, 연월일시분초 중에 값이 이상하게 들어갔을 수도 있어서 제한 걸기
    if not (2000< year and 1 <= month <= 12 and 1 <= day <= 31 and 
            0 <= hour < 24 and 0 <= minute < 60 and 0 <= second < 60):
        return None
    
    #Timestamp 로 date_time 변환
    return pd.Timestamp(
        int(year), int(month), int(day),
        int(hour), int(minute), int(second)
    )

df['start_time'] = df['start_time'].apply(clear_time)
df = df.dropna(subset=['start_time'])

In [12]:
df['start_time']

0      2010-07-21 15:00:35
1      2010-07-21 16:53:45
2      2010-07-21 17:25:40
3      2010-07-21 20:31:05
4      2010-07-21 21:02:56
               ...        
7560   2010-09-30 07:36:45
7561   2010-09-30 08:08:36
7562   2010-09-30 08:48:54
7563   2010-09-30 11:50:17
7564   2010-09-30 12:31:10
Name: start_time, Length: 7412, dtype: datetime64[us]

In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 7412 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   type                 7412 non-null   str           
 1   start_time           7412 non-null   datetime64[us]
 2   ambient_temperature  7412 non-null   int64         
 3   battery_id           7412 non-null   str           
 4   test_id              7412 non-null   int64         
 5   uid                  7412 non-null   int64         
 6   filename             7412 non-null   str           
 7   Capacity             2720 non-null   float64       
 8   Re                   1902 non-null   float64       
 9   Rct                  1902 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(3), str(3)
memory usage: 637.0 KB


- 결측 없던 행들에서 153개행씩 없어짐
- Capacity 49 개행 없어짐
- Re, Rct에서 45개행 없어짐

In [14]:
df['ambient_temperature'].value_counts()

ambient_temperature
24    4554
4     2027
43     384
22     240
44     207
Name: count, dtype: int64

In [15]:
df['battery_id'].value_counts()

battery_id
B0006    607
B0005    607
B0007    607
B0034    477
B0033    477
B0036    477
B0018    312
B0043    268
B0042    268
B0044    268
B0054    246
B0056    245
B0055    245
B0047    182
B0045    182
B0048    182
B0046    182
B0041    160
B0053    133
B0039    117
B0040    117
B0038    117
B0032     96
B0029     96
B0030     96
B0031     96
B0028     77
B0027     77
B0025     77
B0026     77
B0049     61
B0050     61
B0052     61
B0051     61
Name: count, dtype: int64

In [16]:
df['test_id'].value_counts()

test_id
0      34
1      34
2      34
3      34
4      34
       ..
611     3
612     3
613     3
614     3
615     3
Name: count, Length: 615, dtype: int64

In [17]:
import pandas as pd
import os

df_list = []

for i in range(1, 7566):
    file_name = str(i).zfill(5) + '.csv'   # 00001.csv
    
    file_path = os.path.join('data1', file_name)
    
    if os.path.exists(file_path):  # 파일 존재 확인
        temp = pd.read_csv(file_path)
        
        temp['file_name'] = file_name  # 출처 표시
        
        df_list.append(temp)

file_df = pd.concat(df_list, ignore_index=True)

In [18]:
file_df

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,file_name,Sense_current,Battery_current,Current_ratio,Battery_impedance,Rectified_Impedance,Current_charge,Voltage_charge
0,4.246711,0.000252,6.212696,0.0002,0.000,0.000,00001.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.246764,-0.001411,6.234019,0.0002,4.262,9.360,00001.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.039277,-0.995093,6.250255,1.0000,3.465,23.281,00001.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.019506,-0.996731,6.302176,1.0000,3.451,36.406,00001.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4.004763,-0.992845,6.361645,1.0000,3.438,49.625,00001.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7376829,4.214016,0.034080,6.398011,NaN,NaN,10777.672,07565.csv,NaN,NaN,NaN,NaN,NaN,0.0348,4.25
7376830,4.213823,0.034787,6.412317,NaN,NaN,10784.047,07565.csv,NaN,NaN,NaN,NaN,NaN,0.0348,4.25
7376831,4.214100,0.034863,6.407888,NaN,NaN,10790.547,07565.csv,NaN,NaN,NaN,NaN,NaN,0.0348,4.25
7376832,4.213995,0.032502,6.417216,NaN,NaN,10796.969,07565.csv,NaN,NaN,NaN,NaN,NaN,0.0348,4.25
